# Disentangled VAE — Polyphonic Music Style Transfer

Self-contained training notebook for the **DisentangledVAE** on the MAESTRO v3.0.0 dataset.

**Before running:**
1. Set the accelerator to **GPU** (P100 or T4) in notebook settings → Accelerator.
2. Add the MAESTRO dataset via **Add Data** → search `maestro-v3` and attach it.
3. Run all cells top-to-bottom.

**Memory fixes applied (v2):**
- Dataset is **memory-mapped** — the NPZ stays on disk; only requested chunks are read into RAM.
- Preprocessing caps at `MAX_CHUNKS` to bound both disk and decompression memory.
- `num_sanity_val_steps=0` skips the pre-training sanity check (the crash point).
- `BATCH_SIZE` reduced from 64 → 32 to lower GPU peak memory.

**Output:** best `.ckpt` in `/kaggle/working/checkpoints/` — download and drop into `lightning_logs/disentangled_vae/` in your project.

In [ ]:
!pip install -q pytorch-lightning==2.2.4 pretty_midi

In [ ]:
import os
import copy
import warnings
import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f"PyTorch  : {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE — switch accelerator to GPU'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════

# Piano-roll parameters — must match preprocess_vae_data.py and app.py
FS           = 24       # frames per second
CHUNK_SIZE   = 256      # time steps per training sample
RHYTHM_PITCH = 60       # C4 — all pitches collapsed here for the rhythm view
QUANT_STEP   = 0.125    # 16th-note at 120 BPM — quantisation grid for the pitch view

# Composers (must match canonical_composer column in maestro-v3.0.0.csv exactly)
COMPOSERS = [
    "Johann Sebastian Bach",
    "Frédéric Chopin",
    "Ludwig van Beethoven",
    "Franz Schubert",
]

# Memory cap: stop collecting chunks once this many have been accumulated.
# At 256×128 float32, 10 000 chunks × 3 arrays ≈ 3.9 GB on disk (uncompressed).
# The NPZ is memory-mapped so only batches are ever in RAM.
# Raise to 20_000+ for a longer, better-converged run if disk allows.
MAX_CHUNKS = 10_000

# Model architecture
LATENT_DIM_RHYTHM = 64
LATENT_DIM_PITCH  = 64
CHANNELS          = 32

# Training  ← BATCH_SIZE halved from 64 to 32 to reduce GPU peak memory
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
EPOCHS        = 50
BETA_R        = 1.0      # final KL weight for rhythm encoder
BETA_P        = 1.0      # final KL weight for pitch encoder
ANNEAL_STEPS  = 10_000   # steps over which beta ramps 0 → BETA_*
NUM_WORKERS   = 2

# Paths
WORKING_DIR = "/kaggle/working"
NPZ_PATH    = os.path.join(WORKING_DIR, "vae_dataset.npz")
CKPT_DIR    = os.path.join(WORKING_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

print("Config OK.")
print(f"  MAX_CHUNKS={MAX_CHUNKS:,}  BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  AUTO-DETECT MAESTRO DATASET PATH
# ══════════════════════════════════════════════════════════════════

def find_maestro_paths(search_root="/kaggle/input"):
    for dirpath, _, files in os.walk(search_root):
        for fname in files:
            if fname == "maestro-v3.0.0.csv":
                return os.path.join(dirpath, fname), dirpath
    return None, None

MAESTRO_CSV, MAESTRO_ROOT = find_maestro_paths()

# ── Manual override — uncomment and fill in if auto-detect fails ──
# MAESTRO_CSV  = "/kaggle/input/<dataset-name>/maestro-v3.0.0/maestro-v3.0.0.csv"
# MAESTRO_ROOT = "/kaggle/input/<dataset-name>/maestro-v3.0.0"

if MAESTRO_CSV is None:
    raise FileNotFoundError(
        "maestro-v3.0.0.csv not found under /kaggle/input.\n"
        "Attach the MAESTRO dataset via Add Data, or set MAESTRO_CSV/MAESTRO_ROOT manually."
    )

print(f"CSV : {MAESTRO_CSV}")
print(f"Root: {MAESTRO_ROOT}")

df = pd.read_csv(MAESTRO_CSV)
df_filtered = df[df['canonical_composer'].isin(COMPOSERS)]
print(f"\nTotal records: {len(df)}  →  selected composers: {len(df_filtered)}")
print(df_filtered['canonical_composer'].value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  MODEL  —  keep in sync with src/models/disentangled_vae.py
# ══════════════════════════════════════════════════════════════════

class Encoder(nn.Module):
    """Conv2d encoder: (B, 256, 128) → mu, log_var each (B, latent_dim)."""
    def __init__(self, latent_dim=64, channels=32):
        super().__init__()
        C = channels
        self.conv1 = nn.Conv2d(1,   C,   kernel_size=(4,12), stride=(2,2), padding=(1,5))
        self.conv2 = nn.Conv2d(C,   C*2, kernel_size=(4,12), stride=(2,2), padding=(1,5))
        self.conv3 = nn.Conv2d(C*2, C*4, kernel_size=(4, 8), stride=(2,2), padding=(1,3))
        self.conv4 = nn.Conv2d(C*4, C*8, kernel_size=(4, 4), stride=(2,2), padding=(1,1))
        self.fc       = nn.Linear(C*8*16*8, 1024)
        self.fc_mu    = nn.Linear(1024, latent_dim)
        self.fc_lv    = nn.Linear(1024, latent_dim)
        self.relu     = nn.ReLU()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc(x))
        return self.fc_mu(x), self.fc_lv(x)


class Decoder(nn.Module):
    """ConvTranspose decoder: (z_r, z_p) → (B, 256, 128) probabilities."""
    def __init__(self, latent_dim=128, channels=32):
        super().__init__()
        C = channels
        self.fc1     = nn.Linear(latent_dim, 1024)
        self.fc2     = nn.Linear(1024, C*8*16*8)
        self.deconv1 = nn.ConvTranspose2d(C*8, C*4, kernel_size=(4,4),  stride=(2,2), padding=(1,1))
        self.deconv2 = nn.ConvTranspose2d(C*4, C*2, kernel_size=(4,8),  stride=(2,2), padding=(1,3))
        self.deconv3 = nn.ConvTranspose2d(C*2, C,   kernel_size=(4,12), stride=(2,2), padding=(1,5))
        self.deconv4 = nn.ConvTranspose2d(C,   1,   kernel_size=(4,12), stride=(2,2), padding=(1,5))
        self.relu    = nn.ReLU()
        self._C      = C

    def forward(self, z_r, z_p):
        z = torch.cat([z_r, z_p], dim=1)
        x = self.relu(self.fc1(z))
        x = self.relu(self.fc2(x))
        x = x.view(x.size(0), self._C*8, 16, 8)
        x = self.relu(self.deconv1(x))
        x = self.relu(self.deconv2(x))
        x = self.relu(self.deconv3(x))
        x = torch.sigmoid(self.deconv4(x))
        return x.squeeze(1)


class DisentangledVAE(pl.LightningModule):
    def __init__(self, latent_dim_rhythm=64, latent_dim_pitch=64,
                 channels=32, lr=1e-3, beta_r=1.0, beta_p=1.0,
                 anneal_steps=10_000):
        super().__init__()
        self.save_hyperparameters()
        self.encoder_rhythm = Encoder(latent_dim=latent_dim_rhythm, channels=channels)
        self.encoder_pitch  = Encoder(latent_dim=latent_dim_pitch,  channels=channels)
        self.decoder = Decoder(latent_dim=latent_dim_rhythm + latent_dim_pitch, channels=channels)

    def reparameterize(self, mu, lv):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv)

    def forward(self, x_r, x_p):
        mu_r, lv_r = self.encoder_rhythm(x_r)
        mu_p, lv_p = self.encoder_pitch(x_p)
        return self.decoder(self.reparameterize(mu_r, lv_r),
                            self.reparameterize(mu_p, lv_p)), mu_r, lv_r, mu_p, lv_p

    def _loss(self, batch, use_noise=True):
        x_orig, x_r, x_p = batch
        mu_r, lv_r = self.encoder_rhythm(x_r)
        mu_p, lv_p = self.encoder_pitch(x_p)
        z_r = self.reparameterize(mu_r, lv_r) if use_noise else mu_r
        z_p = self.reparameterize(mu_p, lv_p) if use_noise else mu_p
        recon = self.decoder(z_r, z_p)
        B = x_orig.size(0)
        recon_loss = F.binary_cross_entropy(recon.view(B,-1), x_orig.view(B,-1), reduction='sum')
        kl_r = -0.5 * torch.sum(1 + lv_r - mu_r.pow(2) - lv_r.exp())
        kl_p = -0.5 * torch.sum(1 + lv_p - mu_p.pow(2) - lv_p.exp())
        return recon_loss, kl_r, kl_p, mu_r, mu_p, B

    def training_step(self, batch, _):
        recon_loss, kl_r, kl_p, mu_r, mu_p, B = self._loss(batch)
        # KL annealing: beta ramps 0 → 1 over anneal_steps to prevent posterior collapse
        anneal = min(1.0, self.global_step / max(1, self.hparams.anneal_steps))
        loss = (recon_loss + anneal * (self.hparams.beta_r*kl_r + self.hparams.beta_p*kl_p)) / B
        self.log_dict({'train_loss': loss, 'recon': recon_loss/B,
                       'kl_r': kl_r/B, 'kl_p': kl_p/B, 'anneal': anneal},
                      prog_bar=True, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, _):
        recon_loss, kl_r, kl_p, _, _, B = self._loss(batch, use_noise=False)
        val_loss = (recon_loss + kl_r + kl_p) / B
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True, on_step=False)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=self.trainer.estimated_stepping_batches, eta_min=1e-5)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}


# Shape check
with torch.no_grad():
    _m = DisentangledVAE()
    _out = _m(torch.randn(2, 256, 128), torch.randn(2, 256, 128))[0]
    assert _out.shape == (2, 256, 128)
    del _m, _out
print(f"Model OK — {sum(p.numel() for p in DisentangledVAE().parameters()):,} parameters")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  PREPROCESSING
#
#  Saves an UNCOMPRESSED NPZ so the dataset cell can memory-map it.
#  Stops at MAX_CHUNKS to bound both disk usage and RAM during array
#  assembly.  At 10 000 chunks the file is ~3.9 GB on disk; only
#  batch-sized slices are ever loaded into RAM during training.
# ══════════════════════════════════════════════════════════════════

def chunk_piano_roll(pr, size):
    """pr shape (128, T) → list of (size, 128) float32 arrays."""
    pr_t = pr.T
    return [pr_t[i*size:(i+1)*size] for i in range(len(pr_t) // size)]


def process_file(path):
    try:
        pm = pretty_midi.PrettyMIDI(path)

        pr_orig = (pm.get_piano_roll(fs=FS) > 0).astype(np.float32)

        pm_r = copy.deepcopy(pm)
        for inst in pm_r.instruments:
            if not inst.is_drum:
                for note in inst.notes:
                    note.pitch = RHYTHM_PITCH
        pr_r = (pm_r.get_piano_roll(fs=FS) > 0).astype(np.float32)

        pm_p = copy.deepcopy(pm)
        for inst in pm_p.instruments:
            if not inst.is_drum:
                for note in inst.notes:
                    note.start = round(note.start / QUANT_STEP) * QUANT_STEP
                    note.end   = round(note.end   / QUANT_STEP) * QUANT_STEP
                    if note.end <= note.start:
                        note.end = note.start + QUANT_STEP
        pr_p = (pm_p.get_piano_roll(fs=FS) > 0).astype(np.float32)

        co = chunk_piano_roll(pr_orig, CHUNK_SIZE)
        cr = chunk_piano_roll(pr_r,    CHUNK_SIZE)
        cp = chunk_piano_roll(pr_p,    CHUNK_SIZE)
        n  = min(len(co), len(cr), len(cp))
        return co[:n], cr[:n], cp[:n]
    except Exception as e:
        print(f"  [skip] {os.path.basename(path)}: {e}")
        return None, None, None


if os.path.exists(NPZ_PATH):
    print(f"NPZ already exists at {NPZ_PATH} — skipping preprocessing.")
    print("Delete the file and re-run this cell to rebuild from scratch.")
else:
    all_orig, all_r, all_p = [], [], []

    for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Preprocessing"):
        # Respect the chunk cap before loading the next file
        if len(all_orig) >= MAX_CHUNKS:
            print(f"\nReached MAX_CHUNKS={MAX_CHUNKS:,} — stopping early.")
            break

        midi_path = os.path.join(MAESTRO_ROOT, row['midi_filename'])
        if not os.path.exists(midi_path):
            midi_path = os.path.join(MAESTRO_ROOT, os.path.basename(row['midi_filename']))
        if not os.path.exists(midi_path):
            continue

        co, cr, cp = process_file(midi_path)
        if co is not None:
            remaining = MAX_CHUNKS - len(all_orig)
            all_orig.extend(co[:remaining])
            all_r.extend(cr[:remaining])
            all_p.extend(cp[:remaining])

    print(f"\nTotal chunks collected: {len(all_orig):,}")
    assert len(all_orig) > 0, "No chunks — check MAESTRO_ROOT and COMPOSERS."

    # Build arrays in one shot (peak RAM = 3 × N × 256 × 128 × 4 bytes)
    # For N=10 000 that is ~3.9 GB — fine on Kaggle's 13 GB RAM.
    X_orig = np.array(all_orig, dtype=np.float32)
    X_r    = np.array(all_r,    dtype=np.float32)
    X_p    = np.array(all_p,    dtype=np.float32)
    del all_orig, all_r, all_p   # free the list memory immediately

    print(f"Shapes: orig={X_orig.shape}  rhythm={X_r.shape}  pitch={X_p.shape}")

    # Save UNCOMPRESSED so the dataset can memory-map it (mmap_mode requires
    # uncompressed .npy data; np.savez_compressed would break this).
    print(f"Saving uncompressed NPZ to {NPZ_PATH} …")
    np.savez(NPZ_PATH, X_original=X_orig, X_rhythm=X_r, X_pitch=X_p)
    del X_orig, X_r, X_p          # arrays now on disk — release RAM

    size_gb = os.path.getsize(NPZ_PATH) / 1e9
    print(f"Saved ({size_gb:.2f} GB).")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  DATASET  —  memory-mapped so the NPZ stays on disk
# ══════════════════════════════════════════════════════════════════

class VAEDataset(Dataset):
    """
    Loads the NPZ with mmap_mode='r': the file stays on disk and only the
    requested index is paged into RAM.  Total RAM usage is O(batch_size),
    not O(dataset_size).
    """
    def __init__(self, npz_path):
        data = np.load(npz_path, mmap_mode='r')
        # Keep as mmap'd arrays — do NOT call torch.from_numpy here,
        # as that would force the entire file into RAM.
        self._orig   = data['X_original']
        self._rhythm = data['X_rhythm']
        self._pitch  = data['X_pitch']
        print(f"Dataset: {len(self._orig):,} samples  (memory-mapped, no RAM copy)")

    def __len__(self):
        return len(self._orig)

    def __getitem__(self, idx):
        # .copy() reads just this chunk from disk into a contiguous array;
        # torch.from_numpy then wraps it with zero extra copy.
        orig   = torch.from_numpy(self._orig[idx].copy())
        rhythm = torch.from_numpy(self._rhythm[idx].copy())
        pitch  = torch.from_numpy(self._pitch[idx].copy())
        return orig, rhythm, pitch


dataset = VAEDataset(NPZ_PATH)

n_val   = max(1, int(0.1 * len(dataset)))
n_train = len(dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
print(f"Train: {n_train:,}  |  Val: {n_val:,}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  TRAINING
# ══════════════════════════════════════════════════════════════════

model = DisentangledVAE(
    latent_dim_rhythm=LATENT_DIM_RHYTHM,
    latent_dim_pitch=LATENT_DIM_PITCH,
    channels=CHANNELS,
    lr=LEARNING_RATE,
    beta_r=BETA_R,
    beta_p=BETA_P,
    anneal_steps=ANNEAL_STEPS,
)

checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="disentangled_vae-epoch{epoch:02d}-val{val_loss:.1f}",
    monitor="val_loss",
    mode="min",
    save_top_k=2,
    save_last=True,
    auto_insert_metric_name=False,
    verbose=True,
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",
    callbacks=[checkpoint_cb, LearningRateMonitor('step')],
    log_every_n_steps=20,
    val_check_interval=0.5,
    gradient_clip_val=1.0,
    # Skip the pre-training sanity check — it ran 2 val batches before
    # and was the exact point the notebook crashed with OOM.
    num_sanity_val_steps=0,
)

print(f"Training {EPOCHS} epochs, batch={BATCH_SIZE}, anneal over {ANNEAL_STEPS:,} steps")
print(f"Checkpoints → {CKPT_DIR}\n")
trainer.fit(model, train_loader, val_loader)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  VERIFY CHECKPOINT + DOWNLOAD INSTRUCTIONS
# ══════════════════════════════════════════════════════════════════

import glob, matplotlib.pyplot as plt

best_ckpt  = checkpoint_cb.best_model_path
all_ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "*.ckpt")))

print("=" * 60)
print(f"Best checkpoint : {best_ckpt}")
print(f"Best val_loss   : {checkpoint_cb.best_model_score:.4f}")
print("All checkpoints :")
for ck in all_ckpts:
    print(f"  {os.path.basename(ck)}  ({os.path.getsize(ck)/1e6:.0f} MB)")

# Load best and run a forward pass to check the encoder isn't collapsed
loaded = DisentangledVAE.load_from_checkpoint(best_ckpt, map_location="cpu")
loaded.eval()

sample_orig, sample_r, sample_p = val_ds[0]
with torch.no_grad():
    mu_r, lv_r = loaded.encoder_rhythm(sample_r.unsqueeze(0))
    mu_p, lv_p = loaded.encoder_pitch(sample_p.unsqueeze(0))
    recon = loaded.decoder(mu_r, mu_p).squeeze(0).numpy()

print("\nEncoder health check (std >> 0.01 = not collapsed):")
print(f"  z_rhythm  mean={mu_r.mean().item():.4f}  std={mu_r.std().item():.4f}")
print(f"  z_pitch   mean={mu_p.mean().item():.4f}  std={mu_p.std().item():.4f}")
print(f"  decoder output range: [{recon.min():.4f}, {recon.max():.4f}]")

# Piano-roll comparison
orig_np = sample_orig.numpy()
active  = np.where(orig_np.sum(axis=0) > 0)[0]
lo = max(0,   active.min() - 4) if len(active) else 21
hi = min(128, active.max() + 5) if len(active) else 108

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, pr, title, cmap in [
    (axes[0], orig_np,         "Original",        "Blues"),
    (axes[1], recon,           "Reconstruction",  "Oranges"),
    (axes[2], recon >= 0.10,   "Threshold @ 0.10","Reds"),
]:
    ax.imshow(pr[:, lo:hi].T, aspect="auto", origin="lower", cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Time steps")
    ax.set_ylabel("Pitch")
plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("NEXT STEPS")
print("  1. Open the Output panel (right sidebar in Kaggle).")
print("  2. Download the best .ckpt file.")
print("  3. Copy it into  lightning_logs/disentangled_vae/  in your project.")
print("     app.py auto-detects the newest .ckpt in that folder.")
print("=" * 60)